# RnD-6: Safe in-time covariates для CUPAC-style снижения дисперсии

Тонкая витрина поверх пакета `rnd_reports`. Проверяем цепочку
`A/B → Scikit CUPAC A → A+B → A+B+C` и демонстрируем, почему unsafe-признаки нельзя использовать.

- Контекст: `../../docs/06_safe_intime_cupac_context.md`
- План реализации: `../../docs/06_safe_intime_cupac_implementation_plan.md`
- Политика безопасности: `../../docs/feature_safety_policy.md`

Только синтетика с фиксированным seed; реальные данные не используются.

In [1]:
from rnd_reports.synthetic.scenarios import make_synthetic_scenario
from rnd_reports.datasets.adapters import make_synthetic_benchmark_dataset
from rnd_reports.benchmark.protocol import run_benchmark
from rnd_reports.benchmark.reporting import format_results_table, plot_ate_ci
from rnd_reports.feature_safety.diagnostics import diagnose
import matplotlib.pyplot as plt

SEED, N = 11, 8000
scenario = make_synthetic_scenario(n=N, seed=SEED)
bds = make_synthetic_benchmark_dataset(n=N, seed=SEED)
print('истинный ATE =', scenario.true_ate, '| n =', bds.n, '| признаков:', len(bds.feature_columns()))

истинный ATE = 2.8 | n = 8000 | признаков: 14


## Классы признаков и balance/missingness gate

A — pre-treatment (CUPAC); B — expert-safe in-time; C — balance-gated in-time;
D — dag-required (вне estimator); E — mediator; F — leakage; unsafe_demo — только демонстрация.
Ожидаем: A/B/C сбалансированы (gate проходят), D/E/F/unsafe_demo — нет.

In [2]:
diagnose(bds.data, bds.treatment_col, bds.feature_registry)

## Бенчмарк методов

`ab_hypex` (A/B baseline) · `hypex_cupac` (reference/parity) ·
`sklearn_cupac_A` → `+B_linear` → `+B+C_linear` (основная цепочка) · `unsafe_demo_optional` (демонстрация риска).

In [3]:
results = run_benchmark(bds, dataset_name='synthetic_canonical', random_state=SEED)
tbl = format_results_table(results)
tbl[['method', 'ate', 'p_value', 'variance_reduction_vs_ab_pct',
     'incremental_variance_reduction_vs_predecessor_pct', 'feature_groups_used', 'safety_status']]

## ATE ± доверительный интервал по методам

Цепочка должна сужать CI, не сдвигая ATE. `unsafe_demo` сужает CI сильнее всех, но **смещает** ATE — это и есть ловушка.

In [4]:
ax = plot_ate_ci(results, true_ate=scenario.true_ate)
plt.tight_layout()
plt.show()

## Выводы

- Каждая безопасная группа (A → B → C) даёт **дополнительное** снижение дисперсии при несмещённом ATE.
- `hypex_cupac` сопоставим с `sklearn_cupac_A` (parity, reference).
- `unsafe_demo` даёт максимальное «снижение дисперсии», но смещает ATE → **не кандидат** к использованию.
- E/F и не прошедшие gate признаки не попадают в estimator.

Подробности и рекомендации — в `report.md`.